# RegimeFlow Master End-to-End Pipeline
This notebook provides a complete, step-by-step, resume-aware execution pipeline that takes you from raw data to the final reproduction of **Table 1** in the paper.

### Pipeline Objectives:
1. **Environment Alignment**: Set up and verify the conda environment, paths, and system variables.
2. **Diagnostics & Patches**: Apply lazy import patches to fix dependencies (Chronos imports), check for NaNs, and download pretrained parameters.
3. **Training Execution (Resume-Aware)**: Launch/resume all point and probability models. Skips already finished models and handles resuming from checkpoints.
4. **Progress Dashboard**: Monitor SLURM jobs and local training progress in real-time.
5. **Evaluation**: Generate predictions from checkpoints and align prediction files.
6. **Table 1 Reproduction**: Recreate the master overall and stratified performance table (with seed-matching partitioning splits).

**Safe execution**: You can click **"Run All"** to execute the entire pipeline. Completed steps are automatically skipped, and interrupted runs will resume from where they left off.

## Section 1: Environment Alignment & Symlink Fix
This cell unsets conflicting SLURM variables, links the dataset directory, and activates the `mambaseries` Conda environment.

In [1]:
%%bash
# 1. Align directories and dataset paths
if [ ! -d "SysBio-Traj" ] && [ -d "../SysBio-Traj" ]; then
    echo "Linking SysBio-Traj from parent directory..."
    ln -sf ../SysBio-Traj ./SysBio-Traj
fi

# 2. Activate Conda env mambaseries
source ~/miniconda3/bin/activate
eval "$(conda shell.bash hook)"
conda activate mambaseries

# 3. Export Python Path
export PYTHONPATH=$(pwd)
echo "✓ Environment linked and paths aligned!"
python -c "import sys; print('Active Python:', sys.executable)"

✓ Environment linked and paths aligned!
Active Python: /deac/opt/rocky9-noarch/python/3.11.8/bin/python


## Section 2: Pre-flight Verification, Patches & NaN Diagnostics
This step applies import patches (such as the lazy imports patch for Chronos to prevent key errors), verifies optional dependencies, downloads the Chronos pretrained model parameters, and scans all existing checkpoints for NaN values (renaming corrupted checkpoint folders to prevent endless resume loops).

In [2]:
import os
import sys
import glob
import torch
import subprocess
from huggingface_hub import snapshot_download

sys.path.extend(['./', '../'])

# 1. Apply Chronos KeyError Import Fix in models/PointForecasting/_point_base.py
print("=== 1. Checking/Applying Chronos import fix in _point_base.py ===")
point_base_fixed = '''import os
import torch
import sys
import torch.nn as nn
sys.path.extend(['./', '../', '../../'])
from models import Nonstationary_Transformer, DLinear, \\
    PatchTST, iTransformer, TimeMixer, TimeXer

# Lazy imports for optional dependencies
try:
    from models import S_Mamba
except Exception:
    S_Mamba = None

try:
    from models import BiMamba4TS
except Exception:
    BiMamba4TS = None

try:
    from models import Chronos
except Exception:
    Chronos = None


class _PointBaseModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model_dict = {
            'Nonstationary_Transformer': Nonstationary_Transformer,
            'DLinear': DLinear,
            'PatchTST': PatchTST,
            'iTransformer': iTransformer,
            'TimeMixer': TimeMixer,
            'TimeXer': TimeXer,
        }
        if S_Mamba is not None:
            self.model_dict['S_Mamba'] = S_Mamba
        if BiMamba4TS is not None:
            self.model_dict['BiMamba4TS'] = BiMamba4TS
        if Chronos is not None:
            self.model_dict['Chronos'] = Chronos
'''

point_base_paths = [
    "models/PointForecasting/_point_base.py",
    "RegimeFlow/models/PointForecasting/_point_base.py"
]
for p in point_base_paths:
    if os.path.exists(os.path.dirname(p)):
        with open(p, "w", encoding="utf-8") as f:
            f.write(point_base_fixed)
        print(f"  ✓ Verified/Fixed imports in {p}")

# 2. Download Chronos Pretrained Model
print("\n=== 2. Checking Chronos pretrained model ===")
chronos_dir = "Materials/PretrainedModel/Chronos_bolt/20M"
os.makedirs(chronos_dir, exist_ok=True)
if not os.path.exists(os.path.join(chronos_dir, "model.safetensors")):
    print(f"  Downloading amazon/chronos-bolt-base to {chronos_dir}...")
    try:
        snapshot_download(repo_id="amazon/chronos-bolt-base", local_dir=chronos_dir)
        print("  ✓ Chronos download successful!")
    except Exception as e:
        print(f"  ⚠️ HuggingFace download failed: {e}. Evaluation will try downloading dynamically.")
else:
    print(f"  ✓ Pretrained model already exists at {chronos_dir}")

# 3. Check for NaNs in existing checkpoints
print("\n=== 3. Checkpoint NaN Diagnostics ===")
ckpt_dir = "results/checkpoints"
if not os.path.exists(ckpt_dir):
    print("  No checkpoints directory found yet (fresh start).")
else:
    for root, _, files in os.walk(ckpt_dir):
        for file in files:
            if file == "last.ckpt" and "corrupted" not in root:
                path = os.path.join(root, file)
                try:
                    state = torch.load(path, map_location="cpu")
                    has_nan = False
                    for k, v in state.get("state_dict", {}).items():
                        if torch.is_tensor(v) and torch.isnan(v).any():
                            has_nan = True
                            break
                    if has_nan:
                        print(f"  ❌ NaN detected in checkpoint: {path}!")
                        corrupted_path = root + "_corrupted_nan"
                        if not os.path.exists(corrupted_path):
                            print(f"     Renaming checkpoint folder to prevent resume loop.")
                            os.rename(root, corrupted_path)
                    else:
                        print(f"  ✓ Checkpoint {path} is clean.")
                except Exception as e:
                    print(f"  ⚠️ Error loading {path}: {e}")

=== 1. Checking/Applying Chronos import fix in _point_base.py ===
  ✓ Verified/Fixed imports in models/PointForecasting/_point_base.py
  ✓ Verified/Fixed imports in RegimeFlow/models/PointForecasting/_point_base.py

=== 2. Checking Chronos pretrained model ===
  ✓ Pretrained model already exists at Materials/PretrainedModel/Chronos_bolt/20M

=== 3. Checkpoint NaN Diagnostics ===
  ⚠️ Error loading results/checkpoints/BiMamba4TS_seed53/53/last.ckpt: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in th

## Section 3: Training Pipeline (Resume-Aware / Idempotent)
This step schedules training jobs for all models. It checks for completed runs (500 epochs) from both SLURM squeue and local logs, automatically resumes unfinished runs from checkpoints, and skips fully trained models.

It will submit jobs via SLURM (`sbatch`) if a SLURM cluster is detected, otherwise it will run them locally in background waves using `run_all_training_local.sh`.

In [3]:
import os
import glob
import re
import subprocess
import torch

MODELS = [
    # (family, script, name)
    ("ProbabilityForecasting", "exp_train_FM.bash", "RegimeFlow"),
    ("ProbabilityForecasting", "exp_train_FM.bash", "TSFlow_PE"),
    ("ProbabilityForecasting", "exp_train_FM.bash", "TSFlow_PE_regime"),
    ("ProbabilityForecasting", "exp_train_FM.bash", "TSDiff_regime"),
    ("ProbabilityForecasting", "exp_train_FM.bash", "CSDI_regime"),
    ("PointForecasting", "exp_train_Point.bash", "iTransformer"),
    ("PointForecasting", "exp_train_Point.bash", "PatchTST"),
    ("PointForecasting", "exp_train_Point.bash", "DLinear"),
    ("PointForecasting", "exp_train_Point.bash", "S_Mamba"),
    ("PointForecasting", "exp_train_Point.bash", "BiMamba4TS"),
    ("PointForecasting", "exp_train_Point.bash", "TimeMixer"),
    ("PointForecasting", "exp_train_Point.bash", "TimeXer"),
    ("PointForecasting", "exp_train_Point.bash", "NSformer")
]

# 1. Detect environment type
try:
    squeue_check = subprocess.check_output(["squeue", "--version"], stderr=subprocess.DEVNULL)
    has_slurm = True
    print("✓ SLURM cluster environment detected. Using sbatch for scheduling.")
except Exception:
    has_slurm = False
    print("⚠️ No SLURM environment detected. Running models locally in background waves.")

# 2. Check active jobs
active_jobs = []
if has_slurm:
    try:
        squeue_out = subprocess.check_output(["squeue", "-u", os.environ.get("USER", ""), "-o", "%j", "--noheader"], text=True)
        active_jobs = [l.strip() for l in squeue_out.split('\n') if l.strip()]
    except Exception:
        pass
else:
    try:
        ps_out = subprocess.check_output(["ps", "-ef"], text=True)
        for line in ps_out.split('\n'):
            if "train_bioTFM.py" in line:
                for _, _, name in MODELS:
                    if name in line:
                        active_jobs.append(name)
    except Exception:
        pass

# 3. Evaluate completed, running, and pending runs
jobs_to_launch = []
print("\n=== Checking Model Training Status ===")
for family, script, name in MODELS:
    ckpt_path = f"results/checkpoints/{name}_seed53/53/last.ckpt"
    pred_53 = f"results/predictions/{name}_seed53/53/test_predictions.npz"
    pred_53_eval = f"results/predictions/full_{name}_eval/53/test_predictions.npz"
    log_pattern = f"train_{name}*.log"
    logs = glob.glob(log_pattern) + glob.glob(f"slurm_train_{name}*.log") + glob.glob(f"../train_{name}*.log")
    
    is_finished = False
    finish_reason = ""
    is_running = any(name in aj for aj in active_jobs)
    
    # If prediction files exist, it is already completely finished and evaluated
    if os.path.exists(pred_53) or os.path.exists(pred_53_eval):
        is_finished = True
        finish_reason = "seed 53 predictions exist"
    
    if not is_finished and os.path.exists(ckpt_path) and not is_running:
        # Check if the logs confirm completed run (either 500 epochs or early stopping)
        for log in logs:
            try:
                with open(log, "r", encoding="utf-8", errors="ignore") as lf:
                    content = lf.read()
                content_lower = content.lower()
                if "all seeds have finished" in content_lower or "complete (exit 0)" in content_lower:
                    is_finished = True
                    finish_reason = "completed log"
                    break
                elif "epoch 499" in content_lower:
                    is_finished = True
                    finish_reason = "reached max epochs (log)"
                    break
                elif "early stopping" in content_lower or "earlystopping" in content_lower or "trainer was signaled to stop" in content_lower or "signaling trainer to stop" in content_lower:
                    is_finished = True
                    finish_reason = "early stopped (log)"
                    break
            except:
                pass
                
        # If checkpoint exists but we have no logs showing done, we check if epoch was at max or early stopped in checkpoint
        if not is_finished:
            try:
                ckpt_state = torch.load(ckpt_path, map_location='cpu')
                epoch = ckpt_state.get("epoch", 0)
                if epoch >= 499:
                    is_finished = True
                    finish_reason = "reached max epochs (checkpoint)"
                else:
                    # Check early stopping callback state
                    callbacks = ckpt_state.get("callbacks", {})
                    for cb_name, cb_state in callbacks.items():
                        if "EarlyStopping" in cb_name:
                            wait_count = cb_state.get("wait_count", 0)
                            patience = cb_state.get("patience", 9999)
                            if wait_count >= patience:
                                is_finished = True
                                finish_reason = f"early stopped at epoch {epoch} (checkpoint)"
                                break
            except:
                pass
                
    if is_finished:
        print(f"  ✓ {name:20s}: Already finished ({finish_reason}). Skipping.")
    elif is_running:
        print(f"  ⏳ {name:20s}: Currently running. Skipping launch.")
    else:
        extra_args = ""
        if os.path.exists(ckpt_path):
            print(f"  🔄 {name:20s}: Found checkpoint. Resuming from checkpoint.")
            extra_args = f"training.resume_from_checkpoint={ckpt_path}"
        else:
            print(f"  🚀 {name:20s}: Will start training from scratch.")
        jobs_to_launch.append((family, script, name, extra_args))

# 4. Launch runs
if not jobs_to_launch:
    print("\n✓ All models are fully trained and verified!")
else:
    print(f"\nLaunching {len(jobs_to_launch)} training jobs...")
    if has_slurm:
        # Submit to SLURM cluster using sbatch
        for family, script, name, extra in jobs_to_launch:
            cmd = [
                "sbatch",
                "--job-name=train_" + name,
                f"--export=ALL,MODEL={name},TRAIN_SCRIPT={script},EXTRA_ARGS={extra}",
                "scripts/submit_single_model.sbatch"
            ]
            try:
                out = subprocess.check_output(cmd, text=True)
                print(f"  Submitted {name} -> {out.strip()}")
            except Exception as e:
                print(f"  ❌ Error submitting {name}: {e}")
    else:
        # Local background waves (waves of max 4)
        local_script = [
            "#!/bin/bash",
            "# Unified Local Background wave launcher",
            "EPOCHS=500",
            "BATCH_SIZE=256",
            "VAL_EVERY_N_EPOCHS=10",
            "SEED=53",
            "COMMON_EXTRA=\"training.accumulate_grad_batches=4 hardware.precision=16-mixed\"",
            ""
        ]
        # Distribute into waves of 4 GPUs
        for idx, (family, script, name, extra) in enumerate(jobs_to_launch):
            gpu_id = idx % 4
            local_script.append(f"CUDA_VISIBLE_DEVICES={gpu_id} WANDB_MODE=offline bash scripts/{script} $EPOCHS {name} 0 $BATCH_SIZE $VAL_EVERY_N_EPOCHS $SEED $COMMON_EXTRA {extra} > train_{name}_gpu{gpu_id}.log 2>&1 &")
            if gpu_id == 3 or idx == len(jobs_to_launch) - 1:
                local_script.append("wait # Wait for current wave to finish")
                local_script.append("")
        
        launcher_path = "run_all_training_local.sh"
        with open(launcher_path, "w", encoding="utf-8", newline="\n") as f:
            f.write("\n".join(local_script))
        os.chmod(launcher_path, 0o755)
        subprocess.Popen(["bash", launcher_path])
        print(f"  ✓ Wave launcher {launcher_path} executed in background!")
        print("  Monitor training progress in the dashboard cell below.")

✓ SLURM cluster environment detected. Using sbatch for scheduling.

=== Checking Model Training Status ===
  ✓ RegimeFlow          : Already finished (seed 53 predictions exist). Skipping.
  ⏳ TSFlow_PE           : Currently running. Skipping launch.
  ⏳ TSFlow_PE_regime    : Currently running. Skipping launch.
  ⏳ TSDiff_regime       : Currently running. Skipping launch.
  ⏳ CSDI_regime         : Currently running. Skipping launch.
  ⏳ iTransformer        : Currently running. Skipping launch.
  🔄 PatchTST            : Found checkpoint. Resuming from checkpoint.
  🔄 DLinear             : Found checkpoint. Resuming from checkpoint.
  ⏳ S_Mamba             : Currently running. Skipping launch.
  ⏳ BiMamba4TS          : Currently running. Skipping launch.
  ⏳ TimeMixer           : Currently running. Skipping launch.
  ⏳ TimeXer             : Currently running. Skipping launch.
  🔄 NSformer            : Found checkpoint. Resuming from checkpoint.

Launching 3 training jobs...
  Submitted P

## Section 4: Live Dashboard & Progress Tracker
This step monitors the progress of all 16 models. It extracts log details in real-time, displays current epoch progress, losses, and active nodes, and checks the status of SLURM queue and local processes.

In [10]:
import os
import re
import glob
import subprocess
import time
import torch
from datetime import datetime

# 1. Get active jobs from squeue
jobs = []
try:
    squeue_out = subprocess.check_output(
        ["squeue", "-u", os.environ.get("USER", ""), "-o", "%i|%j|%T|%M|%R", "--noheader"],
        stderr=subprocess.DEVNULL, text=True
    )
    for line in squeue_out.strip().split('\n'):
        if line.strip():
            parts = line.strip().split('|')
            if len(parts) >= 5:
                jobs.append({
                    "id": parts[0],
                    "name": parts[1],
                    "state": parts[2],
                    "time": parts[3],
                    "node": parts[4]
                })
except Exception:
    pass

# 2. Get local active processes
local_jobs = []
try:
    ps_out = subprocess.check_output(["ps", "-ef"], text=True)
    for line in ps_out.strip().split('\n'):
        if "train_bioTFM.py" in line and "python" in line:
            model_match = re.search(r'model=[a-zA-Z0-9_/]+/([a-zA-Z0-9_]+)', line)
            if model_match:
                local_jobs.append(model_match.group(1))
except Exception:
    pass

# 3. Discover logs
MODELS = ["BiMamba4TS", "CSDI", "CSDI_regime", "Chronos", "DLinear", "iTransformer", 
          "NSformer", "PatchTST", "RegimeFlow", "S_Mamba", "TimeMixer", "TimeXer", 
          "TSDiff", "TSDiff_regime", "TSFlow_PE", "TSFlow_PE_regime"]

log_files = glob.glob("train_*.log") + glob.glob("slurm_*.log") + glob.glob("../train_*.log")
model_logs = {m: [] for m in MODELS}

for filepath in log_files:
    if not os.path.exists(filepath):
        continue
    try:
        mtime = os.path.getmtime(filepath)
        size = os.path.getsize(filepath)
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            if size < 30000:
                content = f.read()
            else:
                f.seek(0)
                head = f.read(15000)
                f.seek(size - 15000)
                tail = f.read(15000)
                content = head + "\n...\n" + tail
    except Exception:
        continue

    model = None
    filename = os.path.basename(filepath)
    for m in MODELS:
        if m in filename:
            model = m
            break
            
    if not model:
        match = re.search(r'model=(?:PointForecasting|ProbabilityForecasting)/([a-zA-Z0-9_]+)', content)
        if match and match.group(1) in MODELS:
            model = match.group(1)
            
    if not model:
        continue

    clean_content = content.replace('\r', '\n')
    epochs = [int(x) for x in re.findall(r'Epoch\s*:?\s*(\d+)', clean_content)]
    epoch = epochs[-1] if epochs else None
    
    losses = re.findall(r'(?:train/loss|loss)\s*=\s*([0-9.e-]+)', clean_content, re.IGNORECASE)
    loss = losses[-1] if losses else "?"
    
    vals = re.findall(r'(?:val/loss|val_loss|val/val_loss)\s*=\s*([0-9.e-]+)', clean_content, re.IGNORECASE)
    val = vals[-1] if vals else "?"
    
    clean_content_lower = clean_content.lower()
    is_done = "all seeds have finished" in clean_content_lower or "complete (exit 0)" in clean_content_lower or "all training complete" in clean_content_lower or "early stopping" in clean_content_lower or "earlystopping" in clean_content_lower or "trainer was signaled to stop" in clean_content_lower or "signaling trainer to stop" in clean_content_lower

    is_active = False
    for job in jobs:
        if job["id"] in filename:
            is_active = True
            break
            
    model_logs[model].append({
        "file": filename,
        "epoch": epoch,
        "loss": loss,
        "val": val,
        "is_done": is_done,
        "mtime": mtime,
        "is_active": is_active
    })

print(f"======== ACTIVE JOBS ({datetime.now().strftime('%H:%M')}) ========")
if jobs:
    for job in jobs:
        print(f"  Job ID: {job['id']:8s} | {job['name']:16s} {job['state']:8s} {job['time']:10s} {job['node']}")
if local_jobs:
    print(f"  Local active python processes: {', '.join(local_jobs)}")
if not jobs and not local_jobs:
    print("  No active training jobs found.")

print("\n======== TRAINING PROGRESS ========")
print(f"{'MODEL':20s} | {'STATUS':12s} | {'EPOCH':12s} | {'TRAIN LOSS':10s} | {'VAL LOSS':10s} | {'ACTIVE NODE':15s}")
print("-" * 88)

for m in MODELS:
    logs = model_logs[m]
    ckpt_path = f"results/checkpoints/{m}_seed53/53/last.ckpt"
    ckpt_alt_path = f"results/checkpoints/full_{m}_eval/42/last.ckpt"
    has_ckpt = os.path.exists(ckpt_path) or os.path.exists(ckpt_alt_path)
    
    if not logs:
        if has_ckpt:
            is_early_stopped = False
            ckpt_epoch = "?"
            for p in [ckpt_path, ckpt_alt_path]:
                if os.path.exists(p):
                    try:
                        c_state = torch.load(p, map_location='cpu')
                        ep = c_state.get("epoch", 0)
                        ckpt_epoch = str(ep + 1)
                        if ep >= 499:
                            is_early_stopped = True
                        else:
                            cbs = c_state.get("callbacks", {})
                            for cb_n, cb_s in cbs.items():
                                if "EarlyStopping" in cb_n:
                                    w_c = cb_s.get("wait_count", 0)
                                    pat = cb_s.get("patience", 9999)
                                    if w_c >= pat:
                                        is_early_stopped = True
                                        break
                    except:
                        pass
            if is_early_stopped:
                print(f"{m:20s} | {'✓ DONE':12s} | {ckpt_epoch + '/500':12s} | {'-':10s} | {'-':10s} | {'-':15s}")
            else:
                print(f"{m:20s} | {'STOPPED':12s} | {ckpt_epoch + '/500':12s} | {'-':10s} | {'-':10s} | {'-':15s}")
        else:
            print(f"{m:20s} | {'NOT STARTED':12s} | {'0/500':12s} | {'-':10s} | {'-':10s} | {'-':15s}")
        continue
        
    logs.sort(key=lambda x: (x["is_active"], x["mtime"]), reverse=True)
    best_log = logs[0]
    
    running_job = None
    for job in jobs:
        if m.lower() in job["name"].lower() or (best_log["is_active"] and job["id"] in best_log["file"]):
            running_job = job
            break
            
    is_local_active = m in local_jobs
    
    if best_log["is_done"] or (best_log["epoch"] is not None and best_log["epoch"] >= 499):
        status = "✓ DONE"
        epoch_str = f"{best_log['epoch']}/500" if best_log['epoch'] is not None else "500/500"
        node_str = "-"
    elif running_job:
        status = "RUNNING"
        epoch_str = f"{best_log['epoch']}/500" if best_log['epoch'] is not None else "starting..."
        node_str = running_job["node"]
    elif is_local_active:
        status = "RUNNING"
        epoch_str = f"{best_log['epoch']}/500" if best_log['epoch'] is not None else "starting..."
        node_str = "local"
    else:
        status = "STOPPED"
        epoch_str = f"{best_log['epoch']}/500" if best_log['epoch'] is not None else "0/500"
        node_str = "-"
        
    print(f"{m:20s} | {status:12s} | {epoch_str:12s} | {best_log['loss']:10s} | {best_log['val']:10s} | {node_str:15s}")

======== ACTIVE JOBS (00:47) ========
  No active training jobs found.

======== TRAINING PROGRESS ========
MODEL                | STATUS       | EPOCH        | TRAIN LOSS | VAL LOSS   | ACTIVE NODE    
----------------------------------------------------------------------------------------
BiMamba4TS           | STOPPED      | ?/500        | -          | -          | -              
CSDI                 | STOPPED      | ?/500        | -          | -          | -              
CSDI_regime          | STOPPED      | ?/500        | -          | -          | -              
Chronos              | NOT STARTED  | 0/500        | -          | -          | -              
DLinear              | STOPPED      | ?/500        | -          | -          | -              
iTransformer         | STOPPED      | ?/500        | -          | -          | -              
NSformer             | STOPPED      | ?/500        | -          | -          | -              
PatchTST             | STOPPED      | ?/500

## Section 5: Prediction Evaluation & Folder Symlinking
This step automatically identifies models that have finished training but do not have evaluation predictions yet. It runs their evaluation (`eval_model.sh`) in the background, then aligns the directory names with symlinks to allow the table suite to dynamically discover them.

In [8]:
import os
import sys
import glob
import subprocess
import time

MODELS = ["BiMamba4TS", "CSDI", "CSDI_regime", "Chronos", "DLinear", "iTransformer", 
          "NSformer", "PatchTST", "RegimeFlow", "S_Mamba", "TimeMixer", "TimeXer", 
          "TSDiff", "TSDiff_regime", "TSFlow_PE", "TSFlow_PE_regime"]

# 1. Ensure optional packages are present
print("=== 1. Checking optional packages ===")
try:
    import chronos
    print("  ✓ chronos package IS installed.")
except ImportError:
    print("  ❌ chronos package not installed. Installing chronos-forecasting...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "chronos-forecasting", "-q"])
    print("  ✓ chronos installed successfully!")

# 2. Launch missing evaluations
print("\n=== 2. Launching Missing Evaluations ===")
eval_launched = 0
for m in MODELS:
    pred_path_53_eval = f"results/predictions/full_{m}_eval/53/test_predictions.npz"
    pred_path_53_train = f"results/predictions/{m}_seed53/53/test_predictions.npz"
    ckpt_path = f"results/checkpoints/{m}_seed53/53/last.ckpt"
    if m == "Chronos":
        ckpt_path = "results/checkpoints/Chronos"
        os.makedirs("results/checkpoints/Chronos", exist_ok=True)
        
    if not os.path.exists(pred_path_53_eval) and not os.path.exists(pred_path_53_train):
        if os.path.exists(ckpt_path) or m == "Chronos":
            print(f"  🚀 Launching evaluation for {m} on seed 53 in background...")
            cmd = f"nohup bash scripts/eval_model.sh {m} 0 53 > eval_{m}.log 2>&1 &"
            subprocess.Popen(cmd, shell=True)
            eval_launched += 1
        else:
            print(f"  ⚠️ Checkpoint missing for {m} at {ckpt_path}. Cannot evaluate.")
    else:
        print(f"  ✓ Predictions already exist for {m}.")

if eval_launched > 0:
    print(f"\nLaunched {eval_launched} background evaluations. Please wait a few minutes for them to complete.")
    print("Monitor them using: tail -f eval_model.log or watch the dashboard cell above.")

# 3. Symlink/Copy prediction folders for auto-discovery
print("\n=== 3. Aligning Prediction Folders for Auto-Discovery ===")
for m in MODELS:
    src_path = f"results/predictions/full_{m}_eval/53/test_predictions.npz"
    src_alt_path = f"results/predictions/{m}_seed53/53/test_predictions.npz"
    dst_dir = f"results/predictions/{m}_seed53/53"
    dst_file = os.path.join(dst_dir, "test_predictions.npz")
    
    if os.path.exists(dst_file):
        print(f"  ✓ Already aligned: {m}")
    elif os.path.exists(src_path):
        os.makedirs(dst_dir, exist_ok=True)
        try:
            os.symlink(os.path.abspath(src_path), dst_file)
            print(f"  ✓ Symlinked prediction for {m} (from full_eval)")
        except:
            import shutil
            shutil.copy2(src_path, dst_file)
            print(f"  ✓ Copied prediction for {m} (from full_eval)")
    elif os.path.exists(src_alt_path):
        print(f"  ✓ Already aligned (from seed53 direct): {m}")
    else:
        print(f"  ❌ Predictions missing for {m}.")

=== 1. Checking optional packages ===
  ✓ chronos package IS installed.

=== 2. Launching Missing Evaluations ===
  🚀 Launching evaluation for BiMamba4TS on seed 53 in background...
  🚀 Launching evaluation for CSDI on seed 53 in background...
  🚀 Launching evaluation for CSDI_regime on seed 53 in background...
  🚀 Launching evaluation for Chronos on seed 53 in background...
  🚀 Launching evaluation for DLinear on seed 53 in background...
  🚀 Launching evaluation for iTransformer on seed 53 in background...
  🚀 Launching evaluation for NSformer on seed 53 in background...
  🚀 Launching evaluation for PatchTST on seed 53 in background...
  ✓ Predictions already exist for RegimeFlow.
  🚀 Launching evaluation for S_Mamba on seed 53 in background...
  🚀 Launching evaluation for TimeMixer on seed 53 in background...
  🚀 Launching evaluation for TimeXer on seed 53 in background...
  🚀 Launching evaluation for TSDiff on seed 53 in background...
  🚀 Launching evaluation for TSDiff_regime on se

In [12]:
%%bash
grep "Done!" eval_*.log | wc -l

0


In [7]:
%%bash
head -n 35 eval_iTransformer.log

Starting Evaluation for iTransformer
Family:      PointForecasting
Data Type:   PointTrajectory
EMA:         false
Checkpoint:  results/checkpoints/iTransformer_seed53/53/last.ckpt
GPU:         0
[rank: 0] Seed set to 42
Configuration:
save_path_dir: ./results
experiment_name: full_iTransformer_eval
seed: 42
test_only: true
ckpt_path: results/checkpoints/iTransformer_seed53/53/last.ckpt
test_without_ckpt: false
data:
  data_type: PointTrajectory
  OOD: true
  train_AR_dataset: false
  data_dir: ./SysBio-Traj
  load_name: SysBio-Traj_index.csv
  val_ratio: 0.1
  test_ratio: 0.2
  input_window: 96
  output_window: 256
  stride: 16
  transform_type: minmax
  condition_names:
  - bound_min
  - bound_max
  - traj_pattern
  - period


## Section 6: Master Table 1 Reproduction
This step loads all aligned predictions, dynamically matches them with their corresponding train/test split partitioning seed (e.g. seed 53 predictions with the seed 53 OOD test split, seed 42 with seed 42 test split), calculates MSE and MAE across the four regime categories (Complex, Stable, Oscillatory, Monotonic) and overall metrics, and generates the final markdown summary table.

In [6]:
import os
import sys

# Ensure stdout/stderr use UTF-8 encoding (especially on Windows)
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass
if hasattr(sys.stderr, 'reconfigure'):
    try:
        sys.stderr.reconfigure(encoding='utf-8')
    except Exception:
        pass

import numpy as np
import pandas as pd
import torch
from typing import List, Dict, Tuple

# 1. Define the paper's regime categories & evaluation mappings
REGIME_GROUPS = {
    "Complex": [0],
    "Stable": [1, 2],
    "Oscillatory": [3],
    "Monotonic": [4, 5]
}

REGIME_NAMES = {
    0: "Complex (directly_stable)",
    1: "Stable (inc_stable)",
    2: "Stable (dec_stable)",
    3: "Oscillatory (oscillation)",
    4: "Monotonic (increasing)",
    5: "Monotonic (decreasing)"
}

# Mapping of model name/dir to display name and group for Table 1
DISPLAY_TO_MODEL = {
    "DLinear": ["DLinear"],
    "iTransformer": ["iTransformer"],
    "PatchTST": ["PatchTST"],
    "TimeMixer": ["TimeMixer"],
    "TimeXer": ["TimeXer"],
    "SMamba": ["S_Mamba", "SMamba"],
    "BiMamba4TS": ["BiMamba4TS"],
    "NSFormer": ["NSformer", "NSFormer"],
    "Chronos": ["Chronos"],
    "CSDI": ["CSDI"],
    "TSDiff": ["TSDiff"],
    "TSFlow": ["TSFlow_PE", "TSFlow"],
    "Ours": ["RegimeFlow_no_cond", "RegimeFlow_no_regime"],
    "CSDI†": ["CSDI_regime"],
    "TSDiff†": ["TSDiff_regime"],
    "TSFlow†": ["TSFlow_PE_regime"],
    "Ours†": ["RegimeFlow"]
}

MODEL_ORDER = [
    ("Point Forecasting", ["DLinear", "iTransformer", "PatchTST", "TimeMixer", "TimeXer", "SMamba", "BiMamba4TS", "NSFormer", "Chronos"]),
    ("Probabilistic Forecasting w/o Regime Condition", ["CSDI", "TSDiff", "TSFlow", "Ours"]),
    ("Probabilistic Forecasting w/ Regime Condition", ["CSDI†", "TSDiff†", "TSFlow†", "Ours†"])
]

def dataframe_to_markdown(df: pd.DataFrame) -> str:
    """
    Convert a pandas DataFrame to a markdown table string.
    Bypasses the need for the external 'tabulate' library.
    """
    if df.empty:
        return ""
    
    cols = list(df.columns)
    rows = df.values.tolist()
    
    # Calculate column widths
    widths = [len(str(c)) for c in cols]
    for row in rows:
        for i, val in enumerate(row):
            widths[i] = max(widths[i], len(str(val)))
            
    # Format header
    header = "| " + " | ".join(f"{str(cols[i]):<{widths[i]}}" for i in range(len(cols))) + " |"
    separator = "| " + " | ".join("-" * widths[i] for i in range(len(cols))) + " |"
    
    # Format rows
    formatted_rows = []
    for row in rows:
        formatted_rows.append("| " + " | ".join(f"{str(row[i]):<{widths[i]}}" for i in range(len(row))) + " |")
        
    return "\n".join([header, separator] + formatted_rows)

def get_display_name_and_group(model_name: str) -> Tuple[str, str]:
    for group, display_list in MODEL_ORDER:
        for disp in display_list:
            if model_name in DISPLAY_TO_MODEL[disp] or model_name.split('_seed')[0] in DISPLAY_TO_MODEL[disp]:
                return disp, group
    return model_name, "Other"

def parse_args() -> Dict:
    """Simple parser for key=value command line arguments."""
    args = {}
    for arg in sys.argv[1:]:
        if '=' in arg:
            k, v = arg.split('=', 1)
            k = k.strip()
            v = v.strip()
            # Clean boolean values
            if v.lower() == 'true':
                v = True
            elif v.lower() == 'false':
                v = False
            # Clean list values (e.g. condition_names=[a,b])
            elif v.startswith('[') and v.endswith(']'):
                v = [x.strip() for x in v[1:-1].split(',')]
            else:
                # Try to parse as int or float
                try:
                    if '.' in v:
                        v = float(v)
                    else:
                        v = int(v)
                except ValueError:
                    pass
            args[k] = v
    return args

def discover_npz_files(save_path_dir: str) -> Dict[str, Dict[int, str]]:
    """
    Scan save_path_dir recursively for test_predictions.npz.
    Returns: { model_name: { seed: npz_path } }
    """
    discovered = {}
    if not os.path.exists(save_path_dir):
        return discovered
        
    for root, dirs, files in os.walk(save_path_dir):
        for file in files:
            if file == 'test_predictions.npz':
                full_path = os.path.join(root, file)
                norm_path = os.path.normpath(full_path).replace('\\', '/')
                parts = norm_path.split('/')
                
                model_name = None
                seed = None
                
                # Try to extract model name and seed from path parts
                # e.g., results/predictions/RegimeFlow_seed53/53/test_predictions.npz
                # or results/checkpoints/RegimeFlow_seed53/test_predictions.npz
                for part in parts:
                    if '_seed' in part:
                        model_name = part.split('_seed')[0]
                        try:
                            seed_str = part.split('_seed')[1]
                            seed = int(''.join(filter(str.isdigit, seed_str)))
                        except Exception:
                            pass
                        break
                    elif part.startswith('full_') and part.endswith('_eval'):
                        model_name = part[5:-5]
                        break
                        
                if model_name is None and len(parts) >= 2:
                    model_name = parts[-2]
                
                # Check if there is a directory part that is a digit (representing seed)
                for part in parts:
                    if part.isdigit():
                        seed = int(part)
                        
                if model_name:
                    if seed is None:
                        seed = 53  # fallback
                    if model_name not in discovered:
                        discovered[model_name] = {}
                    discovered[model_name][seed] = full_path
                    
    return discovered

def load_dataset_and_patterns(
    seed: int,
    system_info_path: str,
    data_dir: str,
    is_ood: bool,
    input_window: int,
    output_window: int,
    stride: int,
    transform_type: str,
    condition_names: List[str]
) -> np.ndarray:
    """Helper to load and partition the test dataset for a specific seed."""
    system_info = pd.read_csv(system_info_path)
    all_files = [
        os.path.join(data_dir, 'Data', row['model_id'], row['model_name'] + '.csv')
        for _, row in system_info.iterrows()
    ]
    
    # Check if files exist
    existing_files = [f for f in all_files if os.path.exists(f)]
    if not existing_files:
        print(f"[-] Error: CSV trajectory files not found in: {os.path.join(data_dir, 'Data')}")
        return None
        
    if is_ood:
        from sklearn.model_selection import train_test_split
        _, test_list = train_test_split(
            all_files, test_size=0.2, random_state=seed
        )
        
        from dataset.BioSeries import MultiBioSeriesDataset
        test_ds = MultiBioSeriesDataset(
            dataset_dirs=test_list,
            input_window=input_window,
            output_window=output_window,
            stride=stride,
            variable_independent=True,
            transform_type=transform_type,
            condition_names=condition_names
        )
    else:
        from dataset.BioSeries import MultiBioSeriesDataset, MultiBioSeriesDataset_Subset
        from sklearn.model_selection import train_test_split
        
        all_dataset = MultiBioSeriesDataset(
            dataset_dirs=all_files,
            input_window=input_window,
            output_window=output_window,
            stride=stride,
            variable_independent=True,
            transform_type=transform_type,
            condition_names=condition_names
        )
        all_list = list(range(len(all_dataset)))
        _, test_indices = train_test_split(
            all_list, test_size=0.2, random_state=seed
        )
        test_ds = MultiBioSeriesDataset_Subset(all_dataset, test_indices)

    # Build lookup lists for True Regime IDs
    parent_ds = test_ds
    is_subset = False
    if hasattr(test_ds, 'dataset'):
        is_subset = True
        parent_ds = test_ds.dataset.dataset
        indices = test_ds.dataset.indices
    elif hasattr(test_ds, 'subset_indices') and test_ds.subset_indices is not None:
        is_subset = True
        parent_ds = test_ds
        indices = test_ds.subset_indices

    parent_patterns = []
    for base_ds in parent_ds.base_datasets:
        num_vars = base_ds.num_vars
        total_window = base_ds.input_window + base_ds.output_window
        max_start = base_ds.length - total_window
        num_windows = 0
        if max_start >= 0:
            num_windows = (max_start // base_ds.stride) + 1
            
        if num_windows == 0:
            continue
            
        traj_patterns = base_ds.system_condition['traj_pattern'][1:] # skip time column
        
        for var_idx in range(num_vars):
            pat = int(np.round(traj_patterns[var_idx]))
            parent_patterns.extend([pat] * num_windows)
            
    parent_patterns = np.array(parent_patterns)
    
    if is_subset:
        true_patterns = parent_patterns[indices]
    else:
        true_patterns = parent_patterns
        
    print(f"Total test dataset samples indexed for partition seed {seed}: {len(true_patterns)}")
    return true_patterns

def evaluate_model_suite(
    save_path_dir: str = "./results",
    experiment_base_name: str = None,  # if None, discovers/evaluates all
    seeds: List[int] = [42],
    data_dir: str = "./SysBio-Traj",
    load_name: str = "SysBio-Traj_index.csv",
    input_window: int = 96,
    output_window: int = 256,
    stride: int = 16,
    transform_type: str = "minmax",
    condition_names: List[str] = ["bound_min", "bound_max", "traj_pattern", "period"],
    is_ood: bool = True,
    partition_seed: int = 42,
    npz_path: str = None
):
    """
    Evaluate saved test_predictions.npz files across multiple seeds,
    aligning samples with dataset metadata to calculate stratified MSE and MAE.
    """
    print("=" * 70)
    print(f"Beginning Cross-Seed Stratified Evaluation")
    print(f"Data Dir: {data_dir}")
    print(f"Results Dir: {save_path_dir}")
    print("=" * 70)
    
    # 1. Add python path to help load local modules
    if '__file__' in globals():
        project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
    else:
        project_root = os.path.abspath('.')
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
        
    # 2. Check if data directory exists
    system_info_path = os.path.join(data_dir, load_name)
    if not os.path.exists(system_info_path):
        print(f"[-] Error: Dataset index not found at: {system_info_path}")
        print("Please place the SysBio-Traj dataset in the workspace or run with data.data_dir=<path>")
        return None

    # Cache for true_patterns to avoid rebuilding repeatedly for the same partition seed
    patterns_cache = {}
    
    # 4. Resolve which models and files to evaluate
    model_prediction_files = {} # { model_name: { seed: npz_path } }
    
    if npz_path is not None:
        # Directly evaluate a single npz file
        model_name = experiment_base_name if experiment_base_name else "Custom"
        model_prediction_files[model_name] = {seeds[0]: npz_path}
    else:
        # Auto-discover or filter by experiment_base_name
        discovered = discover_npz_files(save_path_dir)
        if experiment_base_name:
            # Single or comma-separated list of models
            model_keys = [m.strip() for m in experiment_base_name.split(',')]
            if len(model_keys) == 1 and model_keys[0].lower() == 'all':
                model_prediction_files = discovered
            else:
                for k in model_keys:
                    if k in discovered:
                        model_prediction_files[k] = discovered[k]
                    else:
                        print(f"[-] Warning: No prediction files discovered for model: {k}")
        else:
            # Default: evaluate all discovered models
            model_prediction_files = discovered

    if not model_prediction_files:
        print("[-] No prediction files found for evaluation.")
        return None

    # 5. Evaluate each model
    all_model_results = {} # { display_name: { category: { mse_list, mae_list } } }

    for model_name, seed_paths in model_prediction_files.items():
        disp_name, group_name = get_display_name_and_group(model_name)
        print(f"\nEvaluating Model: {model_name} (Display: {disp_name})")
        
        seed_metrics = {group: {"mse": [], "mae": []} for group in list(REGIME_GROUPS.keys()) + ["Overall"]}
        
        # Determine which seeds to process for this model
        seeds_to_process = [s for s in seeds if s in seed_paths]
        if not seeds_to_process:
            raise ValueError(
                f"\n[CRITICAL ERROR] No prediction files found for the requested seed(s) {seeds} "
                f"for model '{model_name}'.\n"
                f"Available seeds for this model in results/predictions: {list(seed_paths.keys()) if seed_paths else 'None'}.\n"
                f"This indicates that the evaluation for seed {seeds} was either skipped or failed. "
                f"Please ensure you run evaluation for seed {seeds} first, and check eval_{model_name}.log for errors."
            )
            
        for seed in seeds_to_process:
            path = seed_paths.get(seed, None)
            if path is None:
                continue
                
            if not os.path.exists(path):
                continue
                
            print(f"  [+] Processing seed {seed} prediction from: {path}")
            
            # Load true patterns dynamically using the seed as the partition seed
            if seed not in patterns_cache:
                patterns_cache[seed] = load_dataset_and_patterns(
                    seed=seed,
                    system_info_path=system_info_path,
                    data_dir=data_dir,
                    is_ood=is_ood,
                    input_window=input_window,
                    output_window=output_window,
                    stride=stride,
                    transform_type=transform_type,
                    condition_names=condition_names
                )
                
            true_patterns = patterns_cache[seed]
            if true_patterns is None:
                print(f"  [-] Warning: Failed to load dataset patterns for seed {seed}. Skipping seed.")
                continue
                
            npz_data = np.load(path)
            preds = npz_data['predictions']
            targets = npz_data['targets']
            
            if preds.ndim == 3 and preds.shape[-1] == 1:
                preds = preds.squeeze(-1)
            if targets.ndim == 3 and targets.shape[-1] == 1:
                targets = targets.squeeze(-1)
                
            if len(preds) != len(true_patterns):
                print(f"  [-] WARNING: Predictions count ({len(preds)}) does not match dataset patterns count ({len(true_patterns)}) for seed {seed}.")
                print(f"      Attempting to search for matching partition seed based on predictions length...")
                matched = False
                for alt_seed in [42, 53, 25, 81]:
                    if alt_seed not in patterns_cache:
                        patterns_cache[alt_seed] = load_dataset_and_patterns(
                            seed=alt_seed,
                            system_info_path=system_info_path,
                            data_dir=data_dir,
                            is_ood=is_ood,
                            input_window=input_window,
                            output_window=output_window,
                            stride=stride,
                            transform_type=transform_type,
                            condition_names=condition_names
                        )
                    alt_patterns = patterns_cache[alt_seed]
                    if alt_patterns is not None and len(preds) == len(alt_patterns):
                        print(f"      [+] Found matching partition seed: seed {alt_seed} (length {len(alt_patterns)}). Re-aligning!")
                        true_patterns = alt_patterns
                        matched = True
                        break
                if not matched:
                    print(f"      [-] Could not find any seed split matching prediction length of {len(preds)}.")
                    print(f"      Metrics for each regime category will be completely scrambled!")
                
            min_size = min(len(preds), len(true_patterns))
            preds = preds[:min_size]
            targets = targets[:min_size]
            aligned_patterns = true_patterns[:min_size]
            
            # Group-wise metrics
            for group_cat, p_ids in REGIME_GROUPS.items():
                mask = np.isin(aligned_patterns, p_ids)
                if np.sum(mask) == 0:
                    continue
                
                group_preds = preds[mask]
                group_targets = targets[mask]
                
                mse = np.mean((group_preds - group_targets) ** 2)
                mae = np.mean(np.abs(group_preds - group_targets))
                
                seed_metrics[group_cat]["mse"].append(mse)
                seed_metrics[group_cat]["mae"].append(mae)
                
            # Overall metrics
            overall_mse = np.mean((preds - targets) ** 2)
            overall_mae = np.mean(np.abs(preds - targets))
            seed_metrics["Overall"]["mse"].append(overall_mse)
            seed_metrics["Overall"]["mae"].append(overall_mae)

        # Check if any seed was processed
        processed_seeds_count = len(seed_metrics["Overall"]["mse"])
        if processed_seeds_count == 0:
            print(f"  [-] Warning: No seeds processed for {model_name}. skipping.")
            continue
            
        all_model_results[disp_name] = seed_metrics
        
        # Print individual model table
        individual_results = []
        for group in list(REGIME_GROUPS.keys()) + ["Overall"]:
            mses = seed_metrics[group]["mse"]
            maes = seed_metrics[group]["mae"]
            
            mean_mse = np.mean(mses) if len(mses) > 0 else 0.0
            std_mse = np.std(mses) if len(mses) > 0 else 0.0
            mean_mae = np.mean(maes) if len(maes) > 0 else 0.0
            std_mae = np.std(maes) if len(maes) > 0 else 0.0
            
            mse_str = f"{mean_mse:.6f} ± {std_mse:.6f}" if len(mses) > 1 else f"{mean_mse:.6f}"
            mae_str = f"{mean_mae:.6f} ± {std_mae:.6f}" if len(maes) > 1 else f"{mean_mae:.6f}"
            
            individual_results.append({
                "Regime Category": group,
                "MSE (mean ± std)": mse_str,
                "MAE (mean ± std)": mae_str
            })
        df_ind = pd.DataFrame(individual_results)
        print("\n" + "=" * 70)
        print(f"STRATIFIED BENCHMARK PERFORMANCE: {disp_name} (Processed {processed_seeds_count} seeds)")
        print("=" * 70)
        print(dataframe_to_markdown(df_ind))
        print("=" * 70)

    # 6. Generate Combined Table 1
    if len(all_model_results) == 0:
        print("[-] No models were successfully evaluated.")
        return None

    print("\n" + "=" * 120)
    print("FINAL COMBINED STRATIFIED BENCHMARK PERFORMANCE (TABLE 1 REPRODUCTION)")
    print("=" * 120)

    # Columns of Table 1: Model, Overall MSE, Overall MAE, Complex MSE, Complex MAE, Stable MSE, Stable MAE, Oscillatory MSE, Oscillatory MAE, Monotonic MSE, Monotonic MAE
    headers = ["Model", "Overall MSE", "Overall MAE", "Complex MSE", "Complex MAE", "Stable MSE", "Stable MAE", "Oscillatory MSE", "Oscillatory MAE", "Monotonic MSE", "Monotonic MAE"]
    table_rows = []

    for group_name, display_list in MODEL_ORDER:
        # Add section header
        table_rows.append([f"**{group_name}**"] + [""] * 10)
        
        has_items_in_group = False
        for disp in display_list:
            if disp in all_model_results:
                has_items_in_group = True
                metrics = all_model_results[disp]
                row = [disp]
                
                for cat in ["Overall", "Complex", "Stable", "Oscillatory", "Monotonic"]:
                    mses = metrics[cat]["mse"]
                    maes = metrics[cat]["mae"]
                    
                    mean_mse = np.mean(mses) if len(mses) > 0 else 0.0
                    std_mse = np.std(mses) if len(mses) > 0 else 0.0
                    mean_mae = np.mean(maes) if len(maes) > 0 else 0.0
                    std_mae = np.std(maes) if len(maes) > 0 else 0.0
                    
                    mse_str = f"{mean_mse:.4f}±{std_mse:.3f}" if len(mses) > 1 else f"{mean_mse:.4f}"
                    mae_str = f"{mean_mae:.4f}±{std_mae:.3f}" if len(maes) > 1 else f"{mean_mae:.4f}"
                    row.append(mse_str)
                    row.append(mae_str)
                table_rows.append(row)
                
        # If none of the predefined models in this group were found, search for any discovered models belonging to this group
        if not has_items_in_group:
            # check if there is any other model belonging to this group
            for disp, metrics in all_model_results.items():
                _, g_name = get_display_name_and_group(disp)
                if g_name == group_name and disp not in display_list:
                    row = [disp]
                    for cat in ["Overall", "Complex", "Stable", "Oscillatory", "Monotonic"]:
                        mses = metrics[cat]["mse"]
                        maes = metrics[cat]["mae"]
                        
                        mean_mse = np.mean(mses) if len(mses) > 0 else 0.0
                        std_mse = np.std(mses) if len(mses) > 0 else 0.0
                        mean_mae = np.mean(maes) if len(maes) > 0 else 0.0
                        std_mae = np.std(maes) if len(maes) > 0 else 0.0
                        
                        mse_str = f"{mean_mse:.4f}±{std_mse:.3f}" if len(mses) > 1 else f"{mean_mse:.4f}"
                        mae_str = f"{mean_mae:.4f}±{std_mae:.3f}" if len(maes) > 1 else f"{mean_mae:.4f}"
                        row.append(mse_str)
                        row.append(mae_str)
                    table_rows.append(row)

    # Print markdown table
    df_combined = pd.DataFrame(table_rows, columns=headers)
    markdown_table = dataframe_to_markdown(df_combined)
    print(markdown_table)
    print("=" * 120)

    # Save results to a CSV / MD file under save_path_dir
    os.makedirs(save_path_dir, exist_ok=True)
    summary_md_path = os.path.join(save_path_dir, "benchmark_table_1.md")
    with open(summary_md_path, "w", encoding="utf-8") as f:
        f.write("# Final Benchmark Stratified Performance (Table 1)\n\n")
        f.write(markdown_table)
        f.write("\n")
    print(f"[+] Master benchmark summary table saved to: {summary_md_path}\n")

    return df_combined

# Run the evaluation suite on our predictions targeting ONLY Seed 53
evaluate_model_suite(
    save_path_dir="./results",
    seeds=[53],
    data_dir="./SysBio-Traj",
    load_name="SysBio-Traj_index.csv"
)


Beginning Cross-Seed Stratified Evaluation
Data Dir: ./SysBio-Traj
Results Dir: ./results
[-] No prediction files found for evaluation.
